# Pipeline Financeiro — inform_27 + Danone
Carrega dados de transporte, associa ingressos Danone, enriquece com coordenadas e quilometragem, calcula métricas de rentabilidade e exporta para Parquet (Power BI).

## 1. Configuração e ligação à base de dados

In [1]:
import platform
import sqlite3
import warnings

import pandas as pd

warnings.filterwarnings("ignore")


def get_paths() -> dict:
    """Devolve os caminhos de ficheiros consoante o sistema operativo."""
    sistema = platform.system()

    if sistema == "Windows":
        return {
            "db": r"C:\Users\LISARR\Documents\python\00.DB\2026.db",
            "parquet": r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27_2026_final.parquet",
        }

    if sistema == "Darwin":
        icloud = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
        return {
            "db": f"{icloud}/00_DB/2026.db",
            "parquet": f"{icloud}/inform_27_2026_final.parquet",
        }

    return {
        "db": "2026.db",
        "parquet": "inform_27_2026_final.parquet",
    }


PATHS = get_paths()

with sqlite3.connect(PATHS["db"]) as con:
    df = pd.read_sql_query("SELECT * FROM inform_27_2026", con)

print(f"Linhas carregadas: {len(df):,}")
df.head()


Linhas carregadas: 774,239


,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LOCDES,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem
0,1,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,0.11,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls
1,2,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,0.4,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls
2,3,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,1,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls
3,4,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,1,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls
4,5,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,1.02,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls


## 2. Dados base — `inform_27_2026`

In [2]:
# Conversão de tipos numéricos
colunas_numericas = [
    "INGRESODT", "COSTEDT", "RENTADT", "PALETSDT",
    "PESO_BRUTO", "PALETS", "KM", "KMREALES",
]
for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df["CODEUT"] = df["CODEUT"].astype(str)

# Remover espaços em branco de todas as colunas de texto
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# Filtrar apenas Gestion == "LIS"
df = df[df["GESTION"] == "LIS"]

# Datas (após o trim, sem componente de hora)
df["FCARGA"] = pd.to_datetime(df["FCARGA"], format="%Y%m%d", errors="coerce").dt.date
df["FENTREGA"] = pd.to_datetime(df["FENTREGA"], format="%Y%m%d", errors="coerce").dt.date

df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LOCDES,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem
93,94,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,158.09,-158.09,11.00,...,300390,NO USAR TRANP.FERNANDO SIMÕES MONTEIRO,RFG,EUR,Trailer 33 plts,33,DIESEL,145.559,NO,SAL_DAT027 (1).xls
94,95,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,5.46,-5.46,0.38,...,319558,3. MALAQUIAS - CASH & CARRY O. AZ,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,230.939,NO,SAL_DAT027 (1).xls
95,96,PTG,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,13.3,6.75,6.55,0.47,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,TAM,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (1).xls
96,97,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,15.95,-15.95,1.11,...,319783,"3. MARABUTO-PRODUT.ALIMENTARES,SA",RFGRFG,EUR,Trailer 33 plts,33,DIESEL,204.542,NO,SAL_DAT027 (1).xls
97,98,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,7.33,-7.33,0.51,...,319201,3. COOPERATIVA AGRICOLA DA TOCHA,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,171.983,NO,SAL_DAT027 (1).xls


## 3. Pipeline Danone — cálculo do ingresso por entrega

In [3]:
# Ingressos Danone: carregar e converter peso líquido (formato PT: milhar '.' / decimal ',')
with sqlite3.connect(PATHS["db"]) as con:
    df_danone = pd.read_sql_query("SELECT * FROM ingresso_danone_2026", con)

df_danone.columns = df_danone.columns.str.strip()

df_danone["JDEKNS"] = pd.to_numeric(
    df_danone["JDEKNS"].astype("string").str.strip()
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

df_danone["PCODCL"] = df_danone["PCODCL"].astype("string").str.strip()

df_danone.head()

,id,ENTORNO,PREFPE,PRESPE,PFEENT,CODACT,PCODCL,PNOMCL,PRUTA,PCATCL,...,JDECJS,JDEPLS,JDECPS,JDERLS,JDEKBS,JDEKNS,JDEUNE,DATEXC,UNIDAC,ficheiro_origem
0,1,FGE50PTG,5021758091 413963878,4070599,20260102,011,350150908,PREÇO BAIXO - DOIS AMIGOS,1734,PRE,...,"16,998",",074","1,148",",328","45,375",40.92,0,20251230,66,INFKILP_2026_PTG.CSV
1,2,FGE50PTG,5021765321 413964675,4070604,20260102,011,350151650,PREÇO BAIXO SUP. AV. FER. AROS,1784,PRE,...,"12,248",",058",",841",",254","40,964",37.12,0,20251230,56,INFKILP_2026_PTG.CSV
2,3,FGE50PTG,5021758092 413969049,4070599,20260102,011,350392489,"SANDRA TROVISCO, UNIPESSOAL, L",1734,PRE,...,"7,494",",027",",496",",117","24,670",22.36,0,20251230,45,INFKILP_2026_PTG.CSV
3,4,FGE50PTG,5021742353 413966257,4070599,20260102,011,350194778,PREÇO BAIXO - ANGEIRAS,1734,PRE,...,"10,164",",044",",747",",193","30,749",27.68,0,20251230,59,INFKILP_2026_PTG.CSV
4,5,FGE50PTG,5021756292 413963116,4070599,20260102,011,350390901,FROIZ - BRAGA II- RETAIL CENTE,1734,PRE,...,"39,913",",191","2,504",",850","120,162",108.46,0,20251230,90,INFKILP_2026_PTG.CSV


In [4]:
with sqlite3.connect(PATHS["db"]) as con:
    df_clientes = pd.read_sql_query(
        "SELECT pcodcl, tipo_local FROM danone_clientes_2026",
        con,
    )

df_clientes["pcodcl"] = df_clientes["pcodcl"].astype("string").str.strip()
df_clientes["tipo_local"] = df_clientes["tipo_local"].astype("string").str.strip()

df_danone = df_danone.merge(
    df_clientes,
    left_on="PCODCL",
    right_on="pcodcl",
    how="left",
    validate="many_to_one",
)

print(f"Ingressos sem classificação de cliente: {df_danone['tipo_local'].isna().sum():,}")
df_danone[["PCODCL", "tipo_local"]].head()

Ingressos sem classificação de cliente: 2,185


,PCODCL,tipo_local
0,350150908,Porto Prevenda
1,350151650,Porto Prevenda
2,350392489,Porto Prevenda
3,350194778,Porto Prevenda
4,350390901,Porto Prevenda


In [5]:
df_danone

,id,ENTORNO,PREFPE,PRESPE,PFEENT,CODACT,PCODCL,PNOMCL,PRUTA,PCATCL,...,JDECPS,JDERLS,JDEKBS,JDEKNS,JDEUNE,DATEXC,UNIDAC,ficheiro_origem,pcodcl,tipo_local
0,1,FGE50PTG,5021758091 413963878,4070599,20260102,011,350150908,PREÇO BAIXO - DOIS AMIGOS,1734,PRE,...,"1,148",",328","45,375",40.92,0,20251230,66,INFKILP_2026_PTG.CSV,350150908,Porto Prevenda
1,2,FGE50PTG,5021765321 413964675,4070604,20260102,011,350151650,PREÇO BAIXO SUP. AV. FER. AROS,1784,PRE,...,",841",",254","40,964",37.12,0,20251230,56,INFKILP_2026_PTG.CSV,350151650,Porto Prevenda
2,3,FGE50PTG,5021758092 413969049,4070599,20260102,011,350392489,"SANDRA TROVISCO, UNIPESSOAL, L",1734,PRE,...,",496",",117","24,670",22.36,0,20251230,45,INFKILP_2026_PTG.CSV,350392489,Porto Prevenda
3,4,FGE50PTG,5021742353 413966257,4070599,20260102,011,350194778,PREÇO BAIXO - ANGEIRAS,1734,PRE,...,",747",",193","30,749",27.68,0,20251230,59,INFKILP_2026_PTG.CSV,350194778,Porto Prevenda
4,5,FGE50PTG,5021756292 413963116,4070599,20260102,011,350390901,FROIZ - BRAGA II- RETAIL CENTE,1734,PRE,...,"2,504",",850","120,162",108.46,0,20251230,90,INFKILP_2026_PTG.CSV,350390901,Porto Prevenda
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56067,56069,FGE50AZA,5023616297 415738816,3796042,20260822,011,350420110,PD - ALFENA N/P,1100,DI2,...,"14,000",",006","1518,727",1439.76,0,20260820,0,INFKILP_2026_aza.CSV,<NA>,<NA>
56068,56070,FGE50AZA,5023616298 415738817,3795951,20260821,011,350419876,PD-AZAMBUJA AMBIENTE,2000,DI2,...,"18,000",",009","2251,454",2128.872,0,20260820,0,INFKILP_2026_aza.CSV,<NA>,<NA>
56069,56071,FGE50AZA,5023629841 415746575,3796043,20260822,011,350419883,PD - NÃO PERECIVEIS - ALGOZ,100,DI2,...,"3,177",",000","490,166",467.557,0,20260820,0,INFKILP_2026_aza.CSV,<NA>,<NA>
56070,56072,FGE50AZA,5023629643 415746574,3796043,20260822,011,350419883,PD - NÃO PERECIVEIS - ALGOZ,100,DI2,...,",368",",000","60,277",57.736,0,20260820,0,INFKILP_2026_aza.CSV,<NA>,<NA>


In [6]:
sem_classificacao = df_danone[df_danone["tipo_local"].isna()]

lista_referencias_sem_classificacao = (
    sem_classificacao[["PCODCL", "PNOMCL", "PREFPE", "PFEENT"]]
    .sort_values(["PCODCL", "PFEENT"])
    .reset_index(drop=True)
)

print(f"Linhas sem tipo_local: {len(lista_referencias_sem_classificacao):,}")
lista_referencias_sem_classificacao

Linhas sem tipo_local: 2,185


,PCODCL,PNOMCL,PREFPE,PFEENT
0,350270182,JUMBO GUIMARAES,5023196080 415327864,20260706
1,350270197,PINGO DOCE LINDA A VELHA,5022538870 414719678,20260414
2,350270197,PINGO DOCE LINDA A VELHA,5022923228 415093668,20260529
3,350270200,PINGO DOCE POVOA STO ADRIAO,5023348373 415464189,20260720
4,350270200,PINGO DOCE POVOA STO ADRIAO,5023425480 415542621,20260727
...,...,...,...,...
2180,PRUEBA,CLIENTE PRUEBA,CLIENTE PRUEBA DANONE.19/07/26,20260719
2181,PRUEBA,CLIENTE PRUEBA,Pd azb,20260729
2182,PRUEBA,CLIENTE PRUEBA,CD AZB,20260729
2183,PRUEBA,CLIENTE PRUEBA,prova etiquetas,20260817


In [7]:
with sqlite3.connect(PATHS["db"]) as con:
    df_tarifas = pd.read_sql_query(
        "SELECT tipo_local, tarifa_base FROM Danone_Tarifas_2026",
        con,
    )

df_tarifas["tipo_local"] = df_tarifas["tipo_local"].astype("string").str.strip()
df_tarifas["tarifa_base"] = pd.to_numeric(df_tarifas["tarifa_base"], errors="coerce")

df_danone = df_danone.merge(
    df_tarifas,
    on="tipo_local",
    how="left",
    validate="many_to_one",
)

print(f"Ingressos sem tarifa_base: {df_danone['tarifa_base'].isna().sum():,}")

df_danone.head(5)

Ingressos sem tarifa_base: 19,327


,id,ENTORNO,PREFPE,PRESPE,PFEENT,CODACT,PCODCL,PNOMCL,PRUTA,PCATCL,...,JDERLS,JDEKBS,JDEKNS,JDEUNE,DATEXC,UNIDAC,ficheiro_origem,pcodcl,tipo_local,tarifa_base
0,1,FGE50PTG,5021758091 413963878,4070599,20260102,011,350150908,PREÇO BAIXO - DOIS AMIGOS,1734,PRE,...,",328","45,375",40.92,0,20251230,66,INFKILP_2026_PTG.CSV,350150908,Porto Prevenda,31.96
1,2,FGE50PTG,5021765321 413964675,4070604,20260102,011,350151650,PREÇO BAIXO SUP. AV. FER. AROS,1784,PRE,...,",254","40,964",37.12,0,20251230,56,INFKILP_2026_PTG.CSV,350151650,Porto Prevenda,31.96
2,3,FGE50PTG,5021758092 413969049,4070599,20260102,011,350392489,"SANDRA TROVISCO, UNIPESSOAL, L",1734,PRE,...,",117","24,670",22.36,0,20251230,45,INFKILP_2026_PTG.CSV,350392489,Porto Prevenda,31.96
3,4,FGE50PTG,5021742353 413966257,4070599,20260102,011,350194778,PREÇO BAIXO - ANGEIRAS,1734,PRE,...,",193","30,749",27.68,0,20251230,59,INFKILP_2026_PTG.CSV,350194778,Porto Prevenda,31.96
4,5,FGE50PTG,5021756292 413963116,4070599,20260102,011,350390901,FROIZ - BRAGA II- RETAIL CENTE,1734,PRE,...,",850","120,162",108.46,0,20251230,90,INFKILP_2026_PTG.CSV,350390901,Porto Prevenda,31.96


In [8]:
df_danone["ingresso_danone"] = df_danone["JDEKNS"] / 1000 * df_danone["tarifa_base"]

print(f"Linhas com ingresso_danone calculado: {df_danone['ingresso_danone'].notna().sum():,}")
print(f"Linhas sem ingresso_danone (falta JDEKNS ou tarifa_base): {df_danone['ingresso_danone'].isna().sum():,}")

df_danone[["PCODCL", "tipo_local", "JDEKNS", "tarifa_base", "ingresso_danone"]].head()

Linhas com ingresso_danone calculado: 36,745
Linhas sem ingresso_danone (falta JDEKNS ou tarifa_base): 19,327


,PCODCL,tipo_local,JDEKNS,tarifa_base,ingresso_danone
0,350150908,Porto Prevenda,40.92,31.96,1.307803
1,350151650,Porto Prevenda,37.12,31.96,1.186355
2,350392489,Porto Prevenda,22.36,31.96,0.714626
3,350194778,Porto Prevenda,27.68,31.96,0.884653
4,350390901,Porto Prevenda,108.46,31.96,3.466382


In [9]:
df_danone["DATEXC_data"] = pd.to_datetime(df_danone["DATEXC"], format="%Y%m%d", errors="coerce")
df_danone["mes"] = df_danone["DATEXC_data"].dt.month

soma_por_mes = (
    df_danone
    .groupby("mes")
    .agg(ingresso_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)
soma_por_mes["ingresso_total"] = soma_por_mes["ingresso_total"].round(0).map("{:,.0f}".format)
soma_por_mes

,mes,ingresso_total
0,1.0,"131,368"
1,2.0,"121,687"
2,3.0,"141,950"
3,4.0,"148,878"
4,5.0,"138,283"
5,6.0,"160,579"
6,7.0,"181,448"
7,8.0,"110,718"
8,12.0,"7,386"


In [10]:
import calendar

mes8 = df_danone[df_danone["mes"] == 8]

dias_com_dados = mes8["DATEXC_data"].dt.day.nunique()
dias_total_mes = calendar.monthrange(2026, 8)[1]
total_mes8 = mes8["ingresso_danone"].sum(min_count=1)
media_diaria = total_mes8 / dias_com_dados
projecao_mes8 = media_diaria * dias_total_mes

print(f"Dias com dados em agosto: {dias_com_dados}")
print(f"Dias totais no mês: {dias_total_mes}")
print(f"Total registado até agora: {total_mes8:,.0f}")
print(f"Média diária: {media_diaria:,.0f}")
print(f"Projeção para o mês completo: {projecao_mes8:,.0f}")

Dias com dados em agosto: 18
Dias totais no mês: 31
Total registado até agora: 110,718
Média diária: 6,151
Projeção para o mês completo: 190,681


## 4. Associar o ingresso Danone ao dataframe principal

In [11]:
# Somar ingresso Danone por entrega (PREFPE + PFEENT)
df_ingresso_danone = df_danone[["PREFPE", "PFEENT", "ingresso_danone"]].copy()

# PREFPE fica inteiro (não se divide por espaços) — REFERENCIA no df guarda
# os dois códigos colados no mesmo formato ("5021758091     413963878"),
# por isso a chave de correspondência tem de ficar igual dos dois lados.
df_ingresso_danone["PREFPE"] = df_ingresso_danone["PREFPE"].astype("string").str.strip()
df_ingresso_danone["PFEENT"] = pd.to_datetime(
    df_ingresso_danone["PFEENT"].astype("string").str.strip(), format="%Y%m%d", errors="coerce"
)

df_ingresso_danone = (
    df_ingresso_danone
    .groupby(["PREFPE", "PFEENT"], dropna=False)
    .agg(ingresso_danone_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)

# Preparar chaves no df principal
df["REFERENCIA"] = df["REFERENCIA"].astype("string").str.strip()
df["FENTREGA"] = pd.to_datetime(df["FENTREGA"], errors="coerce")
df["_ordem_original"] = range(len(df))

df = df.merge(
    df_ingresso_danone,
    left_on=["REFERENCIA", "FENTREGA"],
    right_on=["PREFPE", "PFEENT"],
    how="left",
    validate="many_to_one",
).sort_values("_ordem_original").reset_index(drop=True)

# Uma entrega pode ter várias linhas em df (mesma REFERENCIA + FENTREGA);
# o ingresso só é atribuído à última linha, para não o contar em duplicado.
ultima_linha = ~df.duplicated(subset=["REFERENCIA", "FENTREGA"], keep="last")

df["ingresso_danone"] = pd.Series(pd.NA, index=df.index, dtype="Float64")
df.loc[ultima_linha, "ingresso_danone"] = df.loc[ultima_linha, "ingresso_danone_total"]

df = df.drop(columns=["PREFPE", "PFEENT", "ingresso_danone_total", "_ordem_original"])

# Ingresso total = ingresso do transporte (INGRESODT) + ingresso Danone
df["total_ingresso"] = df["INGRESODT"].fillna(0) + df["ingresso_danone"].fillna(0)

diferenca = df_danone["ingresso_danone"].sum(min_count=1) - df["ingresso_danone"].sum(min_count=1)
print(f"Diferença entre ingresso Danone calculado e associado: {diferenca:,.2f}")

Diferença entre ingresso Danone calculado e associado: 183,634.76


In [12]:
df.head(5)

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem,ingresso_danone,total_ingresso
0,94,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,158.09,-158.09,11.00,...,RFG,EUR,Trailer 33 plts,33,DIESEL,145.559,NO,SAL_DAT027 (1).xls,<NA>,0.0
1,95,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,5.46,-5.46,0.38,...,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,230.939,NO,SAL_DAT027 (1).xls,31.332933,31.332933
2,96,PTG,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,13.3,6.75,6.55,0.47,...,TAM,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (1).xls,<NA>,13.3
3,97,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,15.95,-15.95,1.11,...,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,204.542,NO,SAL_DAT027 (1).xls,105.402351,105.402351
4,98,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,7.33,-7.33,0.51,...,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,171.983,NO,SAL_DAT027 (1).xls,44.604654,44.604654


In [13]:
df["mes"] = df["FENTREGA"].dt.month

soma_por_mes_df = (
    df
    .groupby("mes")
    .agg(ingresso_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)
soma_por_mes_df["ingresso_total"] = soma_por_mes_df["ingresso_total"].round(0).map("{:,.0f}".format)
soma_por_mes_df

,mes,ingresso_total
0,1,"45,424"
1,2,"39,756"
2,3,"140,415"
3,4,"145,286"
4,5,"140,676"
5,6,"153,871"
6,7,"174,038"
7,8,"119,196"


In [14]:
resumo_por_atividade_mes = (
    df[df["CODACT"].isin(["011", "013", "311"])]
    .groupby(["CODACT", "mes"])
    .agg(
        soma_ingresodt=("INGRESODT", "sum"),
        soma_ingresso_danone=("ingresso_danone", lambda x: x.sum(min_count=1)),
    )
    .reset_index()
)

resumo_por_atividade_mes["soma_ingresodt"] = resumo_por_atividade_mes["soma_ingresodt"].round(0).map("{:,.0f}".format)
resumo_por_atividade_mes["soma_ingresso_danone"] = resumo_por_atividade_mes["soma_ingresso_danone"].round(0).map("{:,.0f}".format)

resumo_por_atividade_mes

,CODACT,mes,soma_ingresodt,soma_ingresso_danone
0,011,1,"19,609","37,905"
1,011,2,"14,356","32,256"
2,011,3,"82,652","126,101"
3,011,4,"79,390","127,438"
4,011,5,"84,288","125,371"
5,011,6,"80,763","134,166"
6,011,7,"36,231","154,085"
7,011,8,"72,116","103,158"
8,013,1,0,"4,033"
9,013,2,0,"3,960"


## 5. Features derivadas

In [15]:
# Data, semana e dia da semana
df["data"] = pd.to_datetime(df["FCARGA"])
df["week_number"] = df["data"].dt.isocalendar().week
df["week_day"] = df["data"].dt.day_name()
df["mes"] = df["data"].dt.month
df["mes_nome"] = df["data"].dt.month_name()

df[["FCARGA", "week_number", "week_day", "mes_nome", "total_ingresso"]].head()

,FCARGA,week_number,week_day,mes_nome,total_ingresso
0,2026-02-01,5,Sunday,February,0.0
1,2026-02-01,5,Sunday,February,31.332933
2,2026-02-01,5,Sunday,February,13.3
3,2026-02-01,5,Sunday,February,105.402351
4,2026-02-01,5,Sunday,February,44.604654


In [16]:
# Normalização da capacidade do camião (agrupar capacidades equivalentes)
mapeamento_capacidade = {
    4: 6, 5: 6, 6: 6,
    8: 12, 12: 12,
    14: 20, 15: 20, 16: 20, 18: 20, 20: 20,
    22: 24, 24: 24,
    33: 33, 66: 66,
}

df["CAMION_CAPACIDAD_NUM"] = pd.to_numeric(df["CAMION_CAPACIDAD"], errors="coerce")
df["capacidade_norm"] = df["CAMION_CAPACIDAD_NUM"].map(mapeamento_capacidade)

# Capacidade em falta (valor 0): preencher com a capacidade mais comum da mesma rota
linhas_sem_capacidade = df["CAMION_CAPACIDAD_NUM"] == 0
if linhas_sem_capacidade.any():
    capacidade_por_rota = (
        df.loc[df["CAMION_CAPACIDAD_NUM"] > 0]
        .groupby("CODEUT")["capacidade_norm"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.max())
    )
    for rota in df.loc[linhas_sem_capacidade, "CODEUT"].unique():
        if rota in capacidade_por_rota.index:
            df.loc[(df["CODEUT"] == rota) & linhas_sem_capacidade, "capacidade_norm"] = capacidade_por_rota[rota]

df["dados_validos"] = df["capacidade_norm"].notna()
print(f"Linhas com capacidade válida: {df['dados_validos'].sum():,} / {len(df):,}")

Linhas com capacidade válida: 101,904 / 106,736


## 6. Coordenadas geográficas (origem e destino)

In [17]:
with sqlite3.connect(PATHS["db"]) as con:
    coords = pd.read_sql_query(
        "SELECT cp AS CP, point_x AS POINT_X, point_y AS POINT_Y FROM coordenadas",
        con,
    )

coords["CP"] = coords["CP"].astype("string").str.strip()
coords["POINT_X"] = pd.to_numeric(coords["POINT_X"], errors="coerce")
coords["POINT_Y"] = pd.to_numeric(coords["POINT_Y"], errors="coerce")
coords["CP_parte"] = coords["CP"].str.split("-").str[0]

# Centroide por prefixo de código postal — usado como fallback quando o CP completo não tem match
centroid = (
    coords.groupby("CP_parte")[["POINT_X", "POINT_Y"]]
    .mean()
    .reset_index()
    .rename(columns={"POINT_X": "longitude_centroid", "POINT_Y": "latitude_centroid"})
)

coords_origem = coords[["CP", "POINT_X", "POINT_Y"]].rename(
    columns={"CP": "CPOSTAL", "POINT_X": "longitude_origem", "POINT_Y": "latitude_origem"}
)
coords_destino = coords[["CP", "POINT_X", "POINT_Y"]].rename(
    columns={"CP": "CPOSTAD", "POINT_X": "longitude_destino", "POINT_Y": "latitude_destino"}
)

df["CPOSTAL_parte"] = df["CPOSTAL"].str.split("-").str[0]
df["CPOSTAD_parte"] = df["CPOSTAD"].str.split("-").str[0]

df = df.merge(coords_origem, on="CPOSTAL", how="left")
df = df.merge(coords_destino, on="CPOSTAD", how="left")

# Preencher falhas de match com o centroide do prefixo do código postal
centroid_origem = centroid.rename(columns={
    "CP_parte": "CPOSTAL_parte",
    "longitude_centroid": "longitude_origem_c",
    "latitude_centroid": "latitude_origem_c",
})
df = df.merge(centroid_origem, on="CPOSTAL_parte", how="left")
df["longitude_origem"] = df["longitude_origem"].fillna(df["longitude_origem_c"])
df["latitude_origem"] = df["latitude_origem"].fillna(df["latitude_origem_c"])

centroid_destino = centroid.rename(columns={
    "CP_parte": "CPOSTAD_parte",
    "longitude_centroid": "longitude_destino_c",
    "latitude_centroid": "latitude_destino_c",
})
df = df.merge(centroid_destino, on="CPOSTAD_parte", how="left")
df["longitude_destino"] = df["longitude_destino"].fillna(df["longitude_destino_c"])
df["latitude_destino"] = df["latitude_destino"].fillna(df["latitude_destino_c"])

df = df.drop(columns=[
    "CPOSTAL_parte", "CPOSTAD_parte",
    "longitude_origem_c", "latitude_origem_c",
    "longitude_destino_c", "latitude_destino_c",
])

print(f"Matches origem: {df['longitude_origem'].notna().sum():,}")
print(f"Matches destino: {df['longitude_destino'].notna().sum():,}")


Matches origem: 0
Matches destino: 0


## 7. Métricas de rentabilidade

In [18]:
df["custo_por_palete"] = df["COSTEDT"] / df["PALETS"].replace(0, 1)
df["ingresso_por_palete"] = df["total_ingresso"] / df["PALETS"].replace(0, 1)

# Taxa de ocupação por rota (CODEUT): total de paletes da rota vs capacidade do veículo
rota_totais = (
    df.groupby("CODEUT")
    .agg(total_palets=("PALETS", "sum"), capacidade_rota=("capacidade_norm", "first"))
    .reset_index()
)
rota_totais["taxa_rota"] = rota_totais["total_palets"] / rota_totais["capacidade_rota"].replace(0, 1) * 100

df = df.merge(rota_totais[["CODEUT", "taxa_rota", "total_palets"]], on="CODEUT", how="left")
df["taxa_ocupacao"] = df["PALETS"] / df["total_palets"].replace(0, 1) * df["taxa_rota"]
df = df.drop(columns=["taxa_rota", "total_palets"])

df["margem"] = df["total_ingresso"] - df["COSTEDT"]
df["margem_por_palete"] = df["ingresso_por_palete"] - df["custo_por_palete"]

df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,dados_validos,longitude_origem,latitude_origem,longitude_destino,latitude_destino,custo_por_palete,ingresso_por_palete,taxa_ocupacao,margem,margem_por_palete
0,94,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,158.09,-158.09,11.00,...,True,NaN,NaN,NaN,NaN,14.371818,0.0,33.333333,-158.09,-14.371818
1,95,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,5.46,-5.46,0.38,...,True,NaN,NaN,NaN,NaN,5.460000,31.332933,3.030303,25.872933,25.872933
2,96,PTG,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,13.3,6.75,6.55,0.47,...,True,NaN,NaN,NaN,NaN,6.750000,13.3,3.030303,6.55,6.55
3,97,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,15.95,-15.95,1.11,...,True,NaN,NaN,NaN,NaN,7.975000,52.701176,6.060606,89.452351,44.726176
4,98,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,7.33,-7.33,0.51,...,True,NaN,NaN,NaN,NaN,7.330000,44.604654,3.030303,37.274654,37.274654


## 8. Selecionar colunas finais e exportar

In [19]:
colunas_finais = [
    "FENTREGA", "CODEUT", "capacidade_norm", "PALETS", "INGRESODT", "total_ingresso", "COSTEDT",
    "PROV_ORIGEN", "LOCORIGEN", "PROV_DESTINO", "LOCDESTINO", "CPOSTAL", "CPOSTAD",
    "longitude_origem", "latitude_origem", "longitude_destino", "latitude_destino",
    "TRANSPORTISTA", "week_day", "week_number", "mes", "mes_nome", "data",
    "dados_validos", "LOCCAR", "LUGARCARGA", "LOCDES", "LUGARDESCARGA", "TRACTORA",
    "PESO_BRUTO", "CODEDT", "CODACT", "TIPOCLIENTE", "TIPOFLUJO",
    "PROV_ENTREGAR", "PAISENTREGAR", "ACTIVIDAD",
    "custo_por_palete", "ingresso_por_palete", "taxa_ocupacao", "margem", "margem_por_palete",
]

df_sel = df[colunas_finais].copy()
df_sel.head()


,FENTREGA,CODEUT,capacidade_norm,PALETS,INGRESODT,total_ingresso,COSTEDT,PROV_ORIGEN,LOCORIGEN,PROV_DESTINO,...,TIPOCLIENTE,TIPOFLUJO,PROV_ENTREGAR,PAISENTREGAR,ACTIVIDAD,custo_por_palete,ingresso_por_palete,taxa_ocupacao,margem,margem_por_palete
0,2026-02-02,3365589,33.0,11.0,0.0,0.0,158.09,Lisboa,Azambuja,ANADIA,...,None,Directo,ANADIA,PORTUGAL,DANONE PORTUGAL CAPILAR,14.371818,0.0,33.333333,-158.09,-14.371818
1,2026-02-02,3365589,33.0,1.0,0.0,31.332933,5.46,Lisboa,Azambuja,Aveiro,...,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,5.460000,31.332933,3.030303,25.872933,25.872933
2,2026-02-02,3365589,33.0,1.0,13.3,13.3,6.75,Lisboa,Azambuja,Coimbra,...,None,Directo,Coimbra,PORTUGAL,SUMOLCOMPAL MARKETING,6.750000,13.3,3.030303,6.55,6.55
3,2026-02-02,3365589,33.0,2.0,0.0,105.402351,15.95,Lisboa,Azambuja,Aveiro,...,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,7.975000,52.701176,6.060606,89.452351,44.726176
4,2026-02-02,3365589,33.0,1.0,0.0,44.604654,7.33,Lisboa,Azambuja,Coimbra,...,TLD PORTUGAL,Directo,Coimbra,PORTUGAL,DANONE PORTUGAL,7.330000,44.604654,3.030303,37.274654,37.274654


In [20]:
resumo_danone_mes = (
    df[df["CODACT"].isin(["011", "013", "311"])]
    .groupby("mes")
    .agg(
        ingresso=("total_ingresso", "sum"),
        custo=("COSTEDT", "sum"),
    )
    .reset_index()
)
resumo_danone_mes["margem"] = resumo_danone_mes["ingresso"] - resumo_danone_mes["custo"]

for coluna in ["ingresso", "custo", "margem"]:
    resumo_danone_mes[coluna] = resumo_danone_mes[coluna].round(0).map("{:,.0f}".format)

resumo_danone_mes

,mes,ingresso,custo,margem
0,1,"72,393","104,896","-32,503"
1,2,"59,368","96,388","-37,019"
2,3,"251,924","233,984","17,941"
3,4,"251,599","232,189","19,410"
4,5,"240,412","226,165","14,247"
5,6,"251,405","242,858","8,547"
6,7,"231,329","264,121","-32,792"
7,8,"195,507","203,937","-8,430"
8,12,"4,730","6,182","-1,452"


In [21]:
cliente_por_entrega = (
    df_danone
    .assign(
        PREFPE=df_danone["PREFPE"].astype("string").str.strip(),
        PFEENT=pd.to_datetime(df_danone["PFEENT"].astype("string").str.strip(), format="%Y%m%d", errors="coerce"),
    )
    .groupby(["PREFPE", "PFEENT"])
    .agg(PCODCL=("PCODCL", "first"), PNOMCL=("PNOMCL", "first"))
    .reset_index()
)

df_com_cliente = df[df["CODACT"].isin(["011", "311"])].merge(
    cliente_por_entrega,
    left_on=["REFERENCIA", "FENTREGA"],
    right_on=["PREFPE", "PFEENT"],
    how="left",
)

margem_por_cliente = (
    df_com_cliente
    .groupby(["PCODCL", "PNOMCL"], dropna=False)
    .agg(
        ingresso=("total_ingresso", "sum"),
        custo=("COSTEDT", "sum"),
        num_entregas=("REFERENCIA", "size"),
    )
    .reset_index()
)
margem_por_cliente["margem"] = margem_por_cliente["ingresso"] - margem_por_cliente["custo"]

margem_por_cliente_fmt = margem_por_cliente.sort_values("margem").head(20).copy()

for coluna in ["ingresso", "custo", "margem"]:
    margem_por_cliente_fmt[coluna] = margem_por_cliente_fmt[coluna].round(0).map("{:,.0f}".format)

margem_por_cliente_fmt

,PCODCL,PNOMCL,ingresso,custo,num_entregas,margem
847,<NA>,NaN,"29,106","92,749",2393,"-63,642"
785,350481958,AUCHAN ALVERCA ARP I,"8,761","22,858",186,"-14,097"
715,350475931,MERCADONA - ALMEIRIM,"5,199","9,319",43,"-4,120"
826,350486792,VITOR JOSE M. MIRANDA - MEADEL,321,"3,404",156,"-3,083"
28,350140353,AUCHAN - VALONGO,"4,879","7,536",41,"-2,657"
786,350481959,AUCHAN VALONGO ARP I,"6,698","9,260",83,"-2,563"
729,350476819,AUCHAN AZAMBUJA,0,"2,478",76,"-2,478"
499,350404880,BOLAMA - LOJA 2,"4,333","6,706",148,"-2,373"
558,350420108,MODELO - ENTREPOSTO AZAMBUJA A,"9,320","11,573",119,"-2,253"
561,350420110,PD - ALFENA N/P,"17,098","19,026",304,"-1,928"


In [22]:
df_11_311 = df[df["CODACT"].isin(["011", "311"])]

custo_medio_palete = df_11_311["COSTEDT"].sum() / df_11_311["PALETS"].sum()
ingresso_danone_medio_palete = df_11_311["ingresso_danone"].sum(min_count=1) / df_11_311["PALETS"].sum()

print(f"Custo médio por palete: {custo_medio_palete:,.0f}")
print(f"Ingresso Danone médio por palete: {ingresso_danone_medio_palete:,.0f}")

Custo médio por palete: 9
Ingresso Danone médio por palete: 7


In [23]:
tabela_medias = (
    df[df["CODACT"].isin(["011", "311"])]
    .groupby("CODACT")
    .apply(lambda g: pd.Series({
        "custo_medio": g["COSTEDT"].sum() / g["PALETS"].sum(),
        "ingresso_medio": g["ingresso_danone"].sum(min_count=1) / g["PALETS"].sum(),
    }))
    .T
)

tabela_medias = tabela_medias.round(0).map("{:,.0f}".format)
tabela_medias

CODACT,011,311
custo_medio,9,8
ingresso_medio,8,3


In [24]:
# Excluir CODEUT que tenham alguma linha com CODACT == "013"
codeut_com_013 = df.loc[df["CODACT"] == "013", "CODEUT"].unique()
df_sem_013 = df[~df["CODEUT"].isin(codeut_com_013)].copy()

df_sem_013["categoria"] = df_sem_013["CODACT"].map({"011": "011", "311": "311"}).fillna("multicliente")

volume_por_rota_categoria = (
    df_sem_013
    .groupby(["CODEUT", "categoria"])
    .agg(volume=("PALETS", "sum"))
    .reset_index()
)

volume_total_por_rota = (
    df_sem_013
    .groupby("CODEUT")
    .agg(volume_total=("PALETS", "sum"))
    .reset_index()
)

volume_por_rota_categoria = volume_por_rota_categoria.merge(volume_total_por_rota, on="CODEUT")
volume_por_rota_categoria["percentagem"] = (
    volume_por_rota_categoria["volume"] / volume_por_rota_categoria["volume_total"] * 100
)

percentagem_media_por_categoria = (
    volume_por_rota_categoria
    .groupby("categoria")
    .agg(pct_media=("percentagem", "mean"))
    .reset_index()
)
percentagem_media_por_categoria["pct_media"] = percentagem_media_por_categoria["pct_media"].round(1).map("{:,.1f}%".format)

percentagem_media_por_categoria

,categoria,pct_media
0,011,59.7%
1,311,30.6%
2,multicliente,78.0%


In [25]:
df_sel.to_parquet(PATHS["parquet"], index=False)

print(f"Ficheiro exportado: {PATHS['parquet']}")
print(f"Linhas: {len(df_sel):,}")
print(f"Colunas: {len(df_sel.columns)}")
print(f"Tamanho: {df_sel.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Ficheiro exportado: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27_2026_final.parquet
Linhas: 106,736
Colunas: 42
Tamanho: 147.76 MB


In [26]:
# Marcar entregas com ingresso_danone atribuído (fazem match ao Excel Danone)
df["tem_danone"] = df.groupby(["REFERENCIA", "FENTREGA"])["ingresso_danone"].transform(lambda x: x.notna().any())

resumo_danone_mes = (
    df[df["CODACT"].isin(["011"]) & df["tem_danone"]]
    .groupby("mes")
    .agg(
        ingresso=("ingresso_danone", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_danone_mes["margem"] = resumo_danone_mes["ingresso"] - resumo_danone_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_danone_mes[coluna] = resumo_danone_mes[coluna].round(0).map("{:,.0f}".format)

resumo_danone_mes

,mes,ingresso,custo,paletes,peso,margem
0,1,"39,147","37,706","2,898","1,052,892","1,440"
1,2,"33,066","32,867","2,336","867,412",200
2,3,"130,585","133,894","14,712","5,164,183","-3,309"
3,4,"129,279","130,943","14,847","4,655,511","-1,663"
4,5,"122,594","129,713","15,388","4,927,142","-7,120"
5,6,"137,493","134,592","14,738","5,140,873","2,900"
6,7,"149,822","146,996","15,105","5,354,006","2,825"
7,8,"96,279","98,863","12,359","3,364,208","-2,584"
8,12,"2,217","2,513",144,"41,341",-297
